# Preparacao dos dados para CNN

Este notebook implementa o item **(1) Preparacao dos dados para CNN** do artefato da Sprint 3.

Objetivo:
- adequar tensores de entrada para CNN (formato, canais e normalizacao)
- manter a logica modular no `src` e usar o notebook apenas para experimento/documentacao


## Decisoes tecnicas adotadas

1. **Formato de tensor**: `channels_last` -> `(N, H, W, C)`, compatvel com `tf.keras.layers.Conv2D`.
2. **Canais**: cada banda espectral vira um canal da imagem (ex.: ASTER multibanda).
3. **Normalizacao**: `zscore` por canal (media 0 e desvio 1), ajustado somente no treino.
4. **Sem vazamento de dados**: normalizador aprendido no treino e reutilizado em validacao/teste.
5. **Rotulos invalidos**: amostras com label `-1` sao removidas (`drop_invalid_labels=True`).


In [1]:
# adiciona a raiz do repo no sys.path (funciona no VS Code e no Colab)
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / 'src').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

repo_root

PosixPath('/home/inteli/Documentos/3ANO/PROJETO/g01')

In [2]:
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from src.models.cnn_data_prep import prepare_cnn_inputs


In [3]:
SEED = 42
DATASET_PATH = repo_root / 'data' / 'pixels_dataset.csv'
CODES_PATH = repo_root / 'data' / 'extracted_codes.json'

# Para teste rapido local, use amostragem.
# Para o experimento final do artefato, defina USE_SAMPLE = False.
USE_SAMPLE = True
SAMPLE_SIZE = 1500

assert DATASET_PATH.exists(), f'Dataset nao encontrado: {DATASET_PATH}'
assert CODES_PATH.exists(), f'Arquivo de codigos nao encontrado: {CODES_PATH}'

df = pd.read_csv(DATASET_PATH)
if USE_SAMPLE:
    df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=SEED).reset_index(drop=True)

print('Dataset carregado:', DATASET_PATH)
print('Shape do dataframe:', df.shape)


Dataset carregado: /home/inteli/Documentos/3ANO/PROJETO/g01/data/pixels_dataset.csv
Shape do dataframe: (295, 147464)


In [5]:
# Gera labels a partir do extracted_codes e remove labels invalidos (-1).
probe = prepare_cnn_inputs(
    df,
    extracted_codes_path=CODES_PATH,
    normalization='none',
    data_format='channels_last',
    drop_invalid_labels=False,
)

y_all = probe['y']
valid_mask = y_all != -1
df_valid = df.loc[valid_mask].reset_index(drop=True)
y_valid = y_all[valid_mask]

print('Amostras totais:', len(df))
print('Amostras validas:', len(df_valid))
print('Distribuicao de classes (validas):', {int(k): int(v) for k, v in pd.Series(y_valid).value_counts().sort_index().items()})


Amostras totais: 295
Amostras validas: 295
Distribuicao de classes (validas): {0: 179, 1: 116}


In [6]:
# Split estratificado: treino (70%), validacao (15%), teste (15%).
idx = np.arange(len(df_valid))
train_idx, temp_idx, y_train, y_temp = train_test_split(
    idx, y_valid, test_size=0.30, random_state=SEED, stratify=y_valid
)
val_idx, test_idx, y_val, y_test = train_test_split(
    temp_idx, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

df_train = df_valid.iloc[train_idx].reset_index(drop=True)
df_val = df_valid.iloc[val_idx].reset_index(drop=True)
df_test = df_valid.iloc[test_idx].reset_index(drop=True)

print('Train/Val/Test:', len(df_train), len(df_val), len(df_test))


Train/Val/Test: 206 44 45


In [7]:
# Preparacao final para CNN com normalizacao por canal (z-score).
train_out = prepare_cnn_inputs(
    df_train,
    labels=y_train,
    normalization='zscore',
    data_format='channels_last',
)

val_out = prepare_cnn_inputs(
    df_val,
    labels=y_val,
    normalization='zscore',
    normalizer=train_out['normalizer'],
    data_format='channels_last',
)

test_out = prepare_cnn_inputs(
    df_test,
    labels=y_test,
    normalization='zscore',
    normalizer=train_out['normalizer'],
    data_format='channels_last',
)

X_train, y_train = train_out['X'], train_out['y']
X_val, y_val = val_out['X'], val_out['y']
X_test, y_test = test_out['X'], test_out['y']

print('X_train shape:', X_train.shape)
print('X_val shape:  ', X_val.shape)
print('X_test shape: ', X_test.shape)
print('y_train shape:', y_train.shape)


X_train shape: (206, 128, 128, 9)
X_val shape:   (44, 128, 128, 9)
X_test shape:  (45, 128, 128, 9)
y_train shape: (206,)


In [8]:
# Checagens para documentacao do artefato
assert X_train.ndim == 4 and X_val.ndim == 4 and X_test.ndim == 4
assert X_train.shape[-1] == train_out['shape_info']['n_channels']

train_mean_by_channel = X_train.mean(axis=(0, 1, 2))
train_std_by_channel = X_train.std(axis=(0, 1, 2))

print('Media por canal (treino normalizado):')
print(np.round(train_mean_by_channel, 4))
print('Desvio padrao por canal (treino normalizado):')
print(np.round(train_std_by_channel, 4))

print('Faixa X_train:', float(X_train.min()), float(X_train.max()))
print('Formato de entrada esperado pela CNN:', '(N, H, W, C)')


Media por canal (treino normalizado):
[ 0. -0. -0. -0. -0. -0. -0. -0. -0.]
Desvio padrao por canal (treino normalizado):
[1. 1. 1. 1. 1. 1. 1. 1. 1.]
Faixa X_train: -1.754380702972412 7.464064598083496
Formato de entrada esperado pela CNN: (N, H, W, C)


In [9]:
# Salva resumo da preparacao em outputs para rastreabilidade
out_dir = repo_root / 'outputs' / 'a04_cnn_data_prep'
out_dir.mkdir(parents=True, exist_ok=True)

summary = {
    'dataset_path': str(DATASET_PATH),
    'codes_path': str(CODES_PATH),
    'use_sample': USE_SAMPLE,
    'sample_size': int(SAMPLE_SIZE),
    'data_format': 'channels_last',
    'normalization': 'zscore',
    'n_channels': int(train_out['shape_info']['n_channels']),
    'height': int(train_out['shape_info']['height']),
    'width': int(train_out['shape_info']['width']),
    'n_train': int(X_train.shape[0]),
    'n_val': int(X_val.shape[0]),
    'n_test': int(X_test.shape[0]),
    'train_class_balance': {
        str(int(k)): int(v)
        for k, v in pd.Series(y_train).value_counts().sort_index().items()
    },
}

with (out_dir / 'cnn_data_prep_summary.json').open('w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

norm = train_out['normalizer']
if norm and norm.get('method') == 'zscore':
    np.savez_compressed(
        out_dir / 'cnn_normalizer_zscore.npz',
        mean=norm['mean'],
        std=norm['std'],
    )

print('Arquivos salvos em:', out_dir)
print('-', out_dir / 'cnn_data_prep_summary.json')
print('-', out_dir / 'cnn_normalizer_zscore.npz')


Arquivos salvos em: /home/inteli/Documentos/3ANO/PROJETO/g01/outputs/a04_cnn_data_prep
- /home/inteli/Documentos/3ANO/PROJETO/g01/outputs/a04_cnn_data_prep/cnn_data_prep_summary.json
- /home/inteli/Documentos/3ANO/PROJETO/g01/outputs/a04_cnn_data_prep/cnn_normalizer_zscore.npz


## Texto curto para o artefato

Os dados foram convertidos de colunas `pixel_*` para tensores 4D no formato `(N, H, W, C)`, em que `C` representa o numero de bandas espectrais. Essa representacao preserva a estrutura espacial do chip e permite o uso direto de operacoes de convolucao e pooling.

A normalizacao foi feita por canal com z-score, calculando media e desvio padrao apenas no conjunto de treino e reaplicando os mesmos parametros em validacao e teste. Essa estrategia reduz variacoes de escala entre bandas e evita vazamento de informacao entre splits, mantendo o protocolo experimental consistente com a avaliacao do modelo CNN.
